# Lecture: Implementation of TD(0) Policy Evaluation for Blackjack

We want to implement TD(0) policy evaluation for the same `Blackjack-v1` environment used in the Monte Carlo notebook. Unlike Monte Carlo methods, TD(0) updates the value function **online** after every single step, bootstrapping from the current estimate of the next state's value rather than waiting for the full return at the end of an episode.

The TD(0) update rule for state $s$ at time $t$ is:
$$V(s_t) \leftarrow V(s_t) + \alpha \left[ r_{t+1} + \gamma V(s_{t+1}) - V(s_t) \right]$$

where $\alpha$ is the step-size (learning rate) and $r_{t+1} + \gamma V(s_{t+1})$ is called the **TD target**.

In [ ]:
!git clone https://github.com/Fjoelsak/RL.git
!cp RL/04_Model_free_prediction/td_eval_agent.py ./

### Exercises

#### Task 1: Implement TD(0) Policy Evaluation

Open `td_eval_agent.py` and implement the `eval(env, n_episodes, policy)` method of the `TDEvalAgent` class.

Unlike MC, TD(0) does **not** need to wait until the end of an episode. After each step $(s, a, r, s')$, apply the TD(0) update directly to `value_table[s]`.

Use $\gamma = 0.9$ and $\alpha = 0.01$.

In [ ]:
import gymnasium as gym
from td_eval_agent import TDEvalAgent

def simple_policy(observation):
    score, dealer_score, usable_ace = observation
    return 1 if score <= 19 else 0

env = gym.make('Blackjack-v1')

agent = TDEvalAgent(gamma=0.9, alpha=0.01)
value = agent.eval(env, 500000, simple_policy)

env.close()

In [ ]:
agent.plot_blackjack(value)

#### Task 2: Effect of the Step-Size $\alpha$

The step-size $\alpha$ controls how strongly each new experience updates the value estimate. Run TD(0) for 500,000 episodes with $\alpha \in \{0.01, 0.1, 0.5\}$ and compare the resulting value functions visually.

- What happens to the value estimates for large $\alpha$?
- What happens for very small $\alpha$?

In [ ]:
import gymnasium as gym
from td_eval_agent import TDEvalAgent

def simple_policy(observation):
    score, dealer_score, usable_ace = observation
    return 1 if score <= 19 else 0

for alpha in [0.01, 0.1, 0.5]:
    env = gym.make('Blackjack-v1')
    agent = TDEvalAgent(gamma=0.9, alpha=alpha)
    value = agent.eval(env, 500000, simple_policy)
    env.close()

    print(f"alpha = {alpha}")
    agent.plot_blackjack(value)

#### Task 3: TD(0) vs. Monte Carlo

Compare the value functions estimated by TD(0) ($\alpha=0.01$) and first-visit MC (from the previous notebook) after 500,000 episodes using the same `simple_policy`.

First, plot both value functions individually. Then visualise the difference $V_{TD} - V_{MC}$ as a heatmap for both the usable-ace and no-usable-ace cases.

- Where do the estimates differ most?
- TD(0) bootstraps from $V(s')$, while MC uses the full return $G_t$. How does this affect bias and variance of the estimates?

In [ ]:
import gymnasium as gym
from td_eval_agent import TDEvalAgent
from mc_eval_agent import MCEvalAgent

def simple_policy(observation):
    score, dealer_score, usable_ace = observation
    return 1 if score <= 19 else 0

env = gym.make('Blackjack-v1')
td_agent = TDEvalAgent(gamma=0.9, alpha=0.01)
td_value = td_agent.eval(env, 500000, simple_policy)
env.close()

env = gym.make('Blackjack-v1')
mc_agent = MCEvalAgent(gamma=0.9)
mc_value = mc_agent.eval(env, 500000, simple_policy)
env.close()

print("TD(0):")
td_agent.plot_blackjack(td_value)

print("Monte Carlo:")
mc_agent.plot_blackjack(mc_value)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

player_sum = np.arange(12, 22)
dealer_show = np.arange(1, 11)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Difference $V_{TD} - V_{MC}$')

for ax, usable_ace, title in zip(axes, [False, True], ['No usable ace', 'Usable ace']):
    diff = np.zeros((len(player_sum), len(dealer_show)))
    for i, p in enumerate(player_sum):
        for j, d in enumerate(dealer_show):
            diff[i, j] = td_value[p, d, usable_ace] - mc_value[p, d, usable_ace]

    im = ax.imshow(diff, origin='lower', aspect='auto',
                   extent=[0.5, 10.5, 11.5, 21.5],
                   vmin=-0.5, vmax=0.5, cmap='RdBu')
    ax.set_title(title)
    ax.set_xlabel('Dealer showing')
    ax.set_ylabel('Player sum')
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()